In [3]:
import pandas as pd
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download required NLTK resources
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

print("Libraries loaded successfully.")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\srika\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\srika\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\srika\AppData\Roaming\nltk_data...


Libraries loaded successfully.


In [4]:
import pandas as pd
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

print("Libraries loaded successfully.")

Libraries loaded successfully.


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\srika\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\srika\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\srika\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [5]:
# Load project dataset

df = pd.read_csv("../dataset/messages.csv")

print("Dataset shape:", df.shape)

print("\nFirst 5 messages:")
display(df.head())

print("\nPriority distribution:")
print(df["priority"].value_counts())

print("\nCategory distribution:")
print(df["category"].value_counts())

Dataset shape: (1800, 3)

First 5 messages:


,message,priority,category
0,"Regarding the technical workshop, Please prepa...",Low,Academic
1,Reunion committee update - venue options Hi ev...,Medium,Personal
2,"Dad's doctor appointment summary Hi, I went wi...",Medium,Personal
3,Data breach concern - customer PII may be expo...,High,Work/Professional
4,Our exclusive discount is now available. Take ...,Low,Promotional



Priority distribution:
priority
Low       600
Medium    600
High      600
Name: count, dtype: int64

Category distribution:
category
Promotional          568
Personal             318
Academic             290
Other                237
Work/Professional    203
Financial            184
Name: count, dtype: int64


In [6]:
# NLP text preprocessing

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)

    # Keep only letters and spaces
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    # Tokenize
    words = text.split()

    # Remove stopwords and lemmatize
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(words)


# Apply preprocessing
df["clean_message"] = df["message"].apply(clean_text)

# Display original and cleaned messages
display(df[["message", "clean_message"]].head(10))

print("\nText preprocessing completed successfully.")

,message,clean_message
0,"Regarding the technical workshop, Please prepa...",regarding technical workshop please prepare re...
1,Reunion committee update - venue options Hi ev...,reunion committee update venue option hi every...
2,"Dad's doctor appointment summary Hi, I went wi...",dad doctor appointment summary hi went dad car...
3,Data breach concern - customer PII may be expo...,data breach concern customer pii may exposed m...
4,Our exclusive discount is now available. Take ...,exclusive discount available take advantage di...
5,Expense Report Red Flag - Executive Spending H...,expense report red flag executive spending hi ...
6,"Regarding the academic notice, Please complete...",regarding academic notice please complete requ...
7,M&A Due Diligence - Financial Model Review Nee...,due diligence financial model review needed as...
8,Our festival discount is now available. Check ...,festival discount available check available de...
9,Sharing grandma's recipe - the famous banana b...,sharing grandma recipe famous banana bread hey...



Text preprocessing completed successfully.


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF vectorizer
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

# Convert cleaned messages into TF-IDF features
X = tfidf.fit_transform(df["clean_message"])

print("TF-IDF feature extraction completed.")
print("Number of messages:", X.shape[0])
print("Number of features:", X.shape[1])

TF-IDF feature extraction completed.
Number of messages: 1800
Number of features: 5000


In [8]:
from sklearn.model_selection import train_test_split

# Target variables
y_priority = df["priority"]
y_category = df["category"]

# Split data for priority prediction
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X,
    y_priority,
    test_size=0.20,
    random_state=42,
    stratify=y_priority
)

# Split data for category prediction
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X,
    y_category,
    test_size=0.20,
    random_state=42,
    stratify=y_category
)

print("Training and testing split completed.")

print("\nPriority:")
print("Training samples:", X_train_p.shape[0])
print("Testing samples:", X_test_p.shape[0])

print("\nCategory:")
print("Training samples:", X_train_c.shape[0])
print("Testing samples:", X_test_c.shape[0])

Training and testing split completed.

Priority:
Training samples: 1440
Testing samples: 360

Category:
Training samples: 1440
Testing samples: 360


In [9]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

# ============================================================
# 1. NAIVE BAYES
# ============================================================

nb_priority = MultinomialNB()
nb_category = MultinomialNB()

nb_priority.fit(X_train_p, y_train_p)
nb_category.fit(X_train_c, y_train_c)


# ============================================================
# 2. LOGISTIC REGRESSION
# ============================================================

lr_priority = LogisticRegression(max_iter=1000)
lr_category = LogisticRegression(max_iter=1000)

lr_priority.fit(X_train_p, y_train_p)
lr_category.fit(X_train_c, y_train_c)


# ============================================================
# 3. LINEAR SVM
# ============================================================

svm_priority = LinearSVC()
svm_category = LinearSVC()

svm_priority.fit(X_train_p, y_train_p)
svm_category.fit(X_train_c, y_train_c)


print("All 3 ML models trained successfully.")
print("\nModels:")
print("1. Naive Bayes")
print("2. Logistic Regression")
print("3. Linear SVM")

All 3 ML models trained successfully.

Models:
1. Naive Bayes
2. Logistic Regression
3. Linear SVM


In [11]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_model(model, X_test, y_test):
    predictions = model.predict(X_test)

    return {
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, average="weighted", zero_division=0),
        "Recall": recall_score(y_test, predictions, average="weighted", zero_division=0),
        "F1-Score": f1_score(y_test, predictions, average="weighted", zero_division=0)
    }


# ============================================================
# PRIORITY MODELS
# ============================================================

priority_results = {}

priority_results["Naive Bayes"] = evaluate_model(
    nb_priority, X_test_p, y_test_p
)

priority_results["Logistic Regression"] = evaluate_model(
    lr_priority, X_test_p, y_test_p
)

priority_results["Linear SVM"] = evaluate_model(
    svm_priority, X_test_p, y_test_p
)


# ============================================================
# CATEGORY MODELS
# ============================================================

category_results = {}

category_results["Naive Bayes"] = evaluate_model(
    nb_category, X_test_c, y_test_c
)

category_results["Logistic Regression"] = evaluate_model(
    lr_category, X_test_c, y_test_c
)

category_results["Linear SVM"] = evaluate_model(
    svm_category, X_test_c, y_test_c
)


# Convert to DataFrames
priority_results_df = pd.DataFrame(priority_results).T
category_results_df = pd.DataFrame(category_results).T

print("PRIORITY MODEL PERFORMANCE")
display(priority_results_df)

print("\nCATEGORY MODEL PERFORMANCE")
display(category_results_df)

PRIORITY MODEL PERFORMANCE


,Accuracy,Precision,Recall,F1-Score
Naive Bayes,0.908333,0.916490,0.908333,0.909468
Logistic Regression,0.922222,0.925260,0.922222,0.922960
Linear SVM,0.936111,0.937295,0.936111,0.936469



CATEGORY MODEL PERFORMANCE


,Accuracy,Precision,Recall,F1-Score
Naive Bayes,0.975000,0.975349,0.975000,0.974972
Logistic Regression,0.980556,0.981586,0.980556,0.980599
Linear SVM,0.986111,0.986227,0.986111,0.986144


In [12]:
import joblib
import os

# Create models folder if it doesn't exist
os.makedirs("../models", exist_ok=True)

# Save best models
joblib.dump(svm_priority, "../models/priority_model.pkl")
joblib.dump(svm_category, "../models/category_model.pkl")

# Save TF-IDF vectorizer
joblib.dump(tfidf, "../models/tfidf_vectorizer.pkl")

print("Models saved successfully.")
print("\nSaved files:")
print("1. models/priority_model.pkl")
print("2. models/category_model.pkl")
print("3. models/tfidf_vectorizer.pkl")

Models saved successfully.

Saved files:
1. models/priority_model.pkl
2. models/category_model.pkl
3. models/tfidf_vectorizer.pkl


In [13]:
# Final prediction function

def predict_message(message):
    # Clean the input message
    cleaned = clean_text(message)

    # Convert message to TF-IDF
    message_vector = tfidf.transform([cleaned])

    # Predict priority
    priority_prediction = svm_priority.predict(message_vector)[0]

    # Predict category
    category_prediction = svm_category.predict(message_vector)[0]

    return priority_prediction, category_prediction


# Test with a sample message
test_message = """
Please submit your project report before tomorrow morning.
The faculty will review all submissions during the project evaluation.
"""

priority, category = predict_message(test_message)

print("Test Message:")
print(test_message)

print("\nPredicted Priority:", priority)
print("Predicted Category:", category)

Test Message:

Please submit your project report before tomorrow morning.
The faculty will review all submissions during the project evaluation.


Predicted Priority: High
Predicted Category: Work/Professional


In [14]:
# ============================================================
# ACTION RECOMMENDATION ENGINE
# ============================================================

def recommend_action(message, priority, category):
    text = message.lower()

    # High-priority messages
    if priority == "High":

        if category == "Academic":
            return "Complete the academic task immediately and check the deadline."

        elif category == "Financial":
            return "Review the financial issue immediately and take the required action."

        elif category == "Work/Professional":
            return "Review the message immediately and respond or escalate if necessary."

        elif category == "Personal":
            return "Review the message and respond as soon as possible."

        elif category == "Other":
            return "Review the message immediately and take appropriate action."

        else:
            return "Review and respond immediately."

    # Medium-priority messages
    elif priority == "Medium":

        if category == "Academic":
            return "Review the academic message and complete the task before the deadline."

        elif category == "Financial":
            return "Review the financial information and respond when required."

        elif category == "Work/Professional":
            return "Review the message and respond during your normal working time."

        elif category == "Personal":
            return "Review the message and respond when convenient."

        elif category == "Other":
            return "Review the message and take action if required."

        else:
            return "Review the message when convenient."

    # Low-priority messages
    else:

        if category == "Promotional":
            return "No immediate action required. Review or ignore the promotional message."

        elif category == "Personal":
            return "No immediate action required. Respond when convenient."

        elif category == "Academic":
            return "Review the message and complete the task before its deadline."

        elif category == "Work/Professional":
            return "Review when convenient and respond if necessary."

        else:
            return "No immediate action required. Review when convenient."


# ============================================================
# TEST RECOMMENDATION ENGINE
# ============================================================

test_message = """
Please submit your project report before tomorrow morning.
The faculty will review all submissions during the project evaluation.
"""

priority, category = predict_message(test_message)

action = recommend_action(
    test_message,
    priority,
    category
)

print("Message:", test_message)
print("\nPriority:", priority)
print("Category:", category)
print("Recommended Action:", action)

Message: 
Please submit your project report before tomorrow morning.
The faculty will review all submissions during the project evaluation.


Priority: High
Category: Work/Professional
Recommended Action: Review the message immediately and respond or escalate if necessary.


In [15]:
# ============================================================
# CONFIDENCE + IMPORTANT FACTORS
# ============================================================

def get_confidence(model, message_vector):
    # Linear SVM does not directly provide probabilities.
    # decision_function gives the confidence score for each class.
    scores = model.decision_function(message_vector)

    # Convert the highest score into a simple confidence percentage.
    # This is a display-oriented confidence estimate.
    if len(scores.shape) == 1:
        score = max(scores)
    else:
        score = max(scores[0])

    confidence = 1 / (1 + pow(2.71828, -abs(score)))

    return round(confidence * 100, 2)


def get_important_factors(message, priority, category):
    text = message.lower()

    factors = []

    # Urgency indicators
    urgency_words = [
        "urgent", "immediately", "asap",
        "today", "tomorrow", "deadline",
        "important", "critical"
    ]

    for word in urgency_words:
        if word in text:
            factors.append(f"Urgency indicator: '{word}'")

    # Financial indicators
    financial_words = [
        "payment", "loan", "debt",
        "bank", "money", "expense",
        "financial", "invoice"
    ]

    for word in financial_words:
        if word in text:
            factors.append(f"Financial indicator: '{word}'")

    # Academic indicators
    academic_words = [
        "assignment", "project", "exam",
        "faculty", "student", "college",
        "submission", "deadline"
    ]

    for word in academic_words:
        if word in text:
            factors.append(f"Academic indicator: '{word}'")

    # Promotional indicators
    promotional_words = [
        "offer", "discount", "sale",
        "deal", "promotion", "shopping"
    ]

    for word in promotional_words:
        if word in text:
            factors.append(f"Promotional indicator: '{word}'")

    # If no specific factors were detected
    if not factors:
        factors.append(f"Detected category: {category}")
        factors.append(f"Detected priority: {priority}")

    # Keep maximum 5 factors
    return factors[:5]


# ============================================================
# TEST COMPLETE ANALYSIS
# ============================================================

def analyze_message(message):

    cleaned = clean_text(message)

    # TF-IDF transformation
    message_vector = tfidf.transform([cleaned])

    # Predictions
    priority = svm_priority.predict(message_vector)[0]
    category = svm_category.predict(message_vector)[0]

    # Confidence
    priority_confidence = get_confidence(
        svm_priority,
        message_vector
    )

    category_confidence = get_confidence(
        svm_category,
        message_vector
    )

    # Recommendation
    action = recommend_action(
        message,
        priority,
        category
    )

    # Important factors
    factors = get_important_factors(
        message,
        priority,
        category
    )

    return {
        "priority": priority,
        "category": category,
        "priority_confidence": priority_confidence,
        "category_confidence": category_confidence,
        "recommended_action": action,
        "important_factors": factors
    }


# Test complete system

test_message = """
Please submit your machine learning project report
before tomorrow morning. The faculty will review
the project during the evaluation.
"""

result = analyze_message(test_message)

print("========== MESSAGE ANALYSIS ==========")
print("Priority:", result["priority"])
print("Priority Confidence:", result["priority_confidence"], "%")
print("Category:", result["category"])
print("Category Confidence:", result["category_confidence"], "%")
print("Recommended Action:", result["recommended_action"])
print("\nImportant Factors:")

for factor in result["important_factors"]:
    print("-", factor)

========== MESSAGE ANALYSIS ==========
Priority: High
Priority Confidence: 61.56 %
Category: Work/Professional
Category Confidence: 54.27 %
Recommended Action: Review the message immediately and respond or escalate if necessary.

Important Factors:
- Urgency indicator: 'tomorrow'
- Academic indicator: 'project'
- Academic indicator: 'faculty'
